# FP/FN Table by Language and Model

This notebook measures false positives and false negatives for each language and model, using the same file layout/config style as the previous Tukey/MAE notebook.

It outputs:

1. `fp_fn_table_all_variants.csv`: raw TP/TN/FP/FN/FPR/FNR for each `(model, language variant)`.
2. `fp_fn_comparison_table.csv`: baseline-vs-category comparison by `(model, base language)`, including `Delta_FP`, `Delta_FN`, `Delta_FPR`, and `Delta_FNR`.
3. LaTeX versions of both tables.

By default, relevance is binarised as `label >= 2`. Change `RELEVANCE_THRESHOLD` if needed.

In [1]:
# Setup environment path, matching the previous notebook style
import sys
import os
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == "statistics_tests":
    os.chdir(cwd.parents[3])  # Go to project root (anaconda_research_project)
    print(f"Changed working directory to project root: {Path.cwd()}")

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEABORN_ROOT = PROJECT_ROOT / "scripts" / "report" / "seaborn_script"
if str(SEABORN_ROOT) not in sys.path:
    sys.path.insert(0, str(SEABORN_ROOT))

Changed working directory to project root: D:\Work\Research_Project\anaconda_research_project


In [2]:
import re
from typing import Dict, List, Optional, Tuple, Sequence

import pandas as pd

from helpers.lang_profiles import get_langs
from scripts.csv_helpers import bump_field_limit
from helpers.output_writer import write_df

In [3]:
# =========================
# Main config
# =========================
LABELS = [0, 1, 2, 3]

# Binary relevance threshold:
# NIST/LLM label >= 2 is treated as relevant; < 2 is treated as non-relevant.
RELEVANCE_THRESHOLD = 2

# Choose the same language profile as the previous notebook.
LANG_PROFILE = "distract_2"
LANGS: List[str] = get_langs(LANG_PROFILE)

# =========================
# Variant comparison config
# =========================
# These suffixes are WITHOUT the leading underscore.
# Example: eng_qp_rem vs eng_distraction_qp_rem -> use qp_rem and distraction_qp_rem.
CATEGORY_SUFFIX = "distraction_qp_rem"
BASELINE_SUFFIX = "qp_rem"

CATEGORY_LABEL = "Distraction Injection w/ PromptArmor (modified)"
BASELINE_LABEL = "Query Injection w/ PromptArmor (modified)"

# =========================
# Dataset/model config
# =========================
TREC_DL_YEAR = "2022"
LABEL_ROOT = Path("outputs/llm_label") / f"trec_dl_{TREC_DL_YEAR}"

# Only include these models. This excludes llama3-8b-instruct.
ALLOWED_MODELS = {
    "gpt-oss-20b",
    "qwen3-32b-v1",
}

KEY_COLS = ["qid", "pid"]
INVALID_CSV = (SEABORN_ROOT / "statistics_tests") / f"invalid_{TREC_DL_YEAR}.csv"

COMPARE_NAME = f"{BASELINE_SUFFIX}_vs_{CATEGORY_SUFFIX}"
OUT_DIR = (
    Path("figures")
    / TREC_DL_YEAR
    / "fp_fn"
    / f"all_models_all_{LANG_PROFILE}_{COMPARE_NAME}_threshold_{RELEVANCE_THRESHOLD}"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ALL_CSV = OUT_DIR / "fp_fn_table_all_variants.csv"
OUT_ALL_TEX = OUT_DIR / "fp_fn_table_all_variants.tex"
OUT_COMPARE_CSV = OUT_DIR / "fp_fn_comparison_table.csv"
OUT_COMPARE_TEX = OUT_DIR / "fp_fn_comparison_table.tex"

print(f"Loaded {len(LANGS)} languages from LANG_PROFILE='{LANG_PROFILE}'")
print(f"Comparing baseline '*_{BASELINE_SUFFIX}' against category '*_{CATEGORY_SUFFIX}'")
print(f"Binary relevance threshold: label >= {RELEVANCE_THRESHOLD}")

Loaded 17 languages from LANG_PROFILE='distract_2'
Comparing baseline '*_qp_rem' against category '*_distraction_qp_rem'
Binary relevance threshold: label >= 2


In [4]:
def suffix_forms(suffix: str) -> Tuple[str, ...]:
    """Return suffix forms with and without the leading underscore."""
    suffix = str(suffix).strip().lower().lstrip("_")
    if not suffix:
        return tuple()
    return (f"_{suffix}", suffix)


CATEGORY_SUFFIXES: Sequence[str] = suffix_forms(CATEGORY_SUFFIX)
BASELINE_SUFFIXES: Sequence[str] = suffix_forms(BASELINE_SUFFIX)


def endswith_any_suffix(s: str, suffixes: Sequence[str]) -> bool:
    s = str(s).strip().lower()
    return any(s.endswith(str(suf).lower()) for suf in suffixes)


def strip_variant_suffixes(lang: str, suffixes: Sequence[str]) -> str:
    s = str(lang).strip().lower()
    # longest suffix first prevents partial stripping problems
    for suf in sorted(suffixes, key=len, reverse=True):
        suf = str(suf).strip().lower()
        if suf and s.endswith(suf):
            return s[: -len(suf)].rstrip("_")
    return s


def base_lang_from_variant(lang: str) -> str:
    """Collapse eng_qp_rem and eng_distraction_qp_rem into eng."""
    return strip_variant_suffixes(
        lang,
        list(CATEGORY_SUFFIXES) + list(BASELINE_SUFFIXES),
    )


def variant_type(lang: str) -> str:
    if endswith_any_suffix(lang, BASELINE_SUFFIXES):
        return "baseline"
    if endswith_any_suffix(lang, CATEGORY_SUFFIXES):
        return "category"
    return "other"

In [5]:
def find_llm_files() -> Dict[str, List[Path]]:
    """
    Returns model -> list of label CSVs.
    Expected layout:
      outputs/llm_label/trec_dl_<YEAR>/<MODEL>/<MODEL>_trecdl_<YEAR>_<LANG>_labels.csv
    """
    if not LABEL_ROOT.exists():
        raise FileNotFoundError(f"LABEL_ROOT not found: {LABEL_ROOT}")

    model_files: Dict[str, List[Path]] = {}
    for model_dir in LABEL_ROOT.iterdir():
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        if model_name not in ALLOWED_MODELS:
            continue

        csv_files = list(model_dir.glob(f"{model_name}_trecdl_{TREC_DL_YEAR}_*_labels.csv"))
        if csv_files:
            model_files[model_name] = csv_files
        else:
            print(f"Warning: no label CSVs found for model {model_name} in {model_dir}")

    return model_files


def get_lang_from_filename(file_path: Path, model: str) -> Optional[str]:
    fname = file_path.name
    pattern = rf"^{re.escape(model)}_trecdl_\d{{4}}_(.+?)_labels\.csv$"
    match = re.search(pattern, fname)
    return match.group(1) if match else None


def load_invalid_keys(path: Path) -> set[tuple[int, str]]:
    if not path.exists():
        print(f"[INFO] No invalid file found: {path}")
        return set()

    inv = pd.read_csv(path)
    if not set(KEY_COLS).issubset(inv.columns):
        raise ValueError(f"{path} must contain columns {KEY_COLS}")

    inv = inv.dropna(subset=KEY_COLS).copy()
    inv["qid"] = pd.to_numeric(inv["qid"], errors="coerce")
    inv = inv.dropna(subset=["qid"])
    inv["qid"] = inv["qid"].astype(int)
    inv["pid"] = inv["pid"].astype(str)

    keys = set(inv[KEY_COLS].drop_duplicates().itertuples(index=False, name=None))
    print(f"[INFO] Loaded {len(keys)} invalid keys from {path}")
    return keys


def load_labels(file_path: Path, invalid_keys: set[tuple[int, str]]) -> pd.DataFrame:
    bump_field_limit()
    df = pd.read_csv(file_path)

    requisite = {"qid", "pid", "relevance", "llm_relevance"}
    missing = requisite - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns {sorted(missing)} in {file_path}")

    df["NIST"] = pd.to_numeric(df["relevance"], errors="coerce")
    df["LLM"] = pd.to_numeric(df["llm_relevance"], errors="coerce")

    df = df.dropna(subset=["NIST", "LLM", "qid", "pid"]).copy()
    df["NIST"] = df["NIST"].astype(int)
    df["LLM"] = df["LLM"].astype(int)
    df = df[df["NIST"].isin(LABELS) & df["LLM"].isin(LABELS)].copy()

    df["qid"] = pd.to_numeric(df["qid"], errors="coerce")
    df = df.dropna(subset=["qid"]).copy()
    df["qid"] = df["qid"].astype(int)
    df["pid"] = df["pid"].astype(str)

    if invalid_keys:
        keys = pd.Index(list(zip(df["qid"].to_numpy(), df["pid"].to_numpy())))
        df = df[~keys.isin(invalid_keys)].copy()

    # Avoid duplicate (qid,pid) rows affecting counts.
    df = df.drop_duplicates(subset=["qid", "pid"]).copy()
    return df

In [6]:
def compute_fp_fn_metrics(df: pd.DataFrame, threshold: int = RELEVANCE_THRESHOLD) -> dict:
    """
    Compute binary relevance confusion counts and rates.

    Gold positive: NIST >= threshold
    Predicted positive: LLM >= threshold
    """
    gold_bin = (df["NIST"] >= threshold).astype(int)
    llm_bin = (df["LLM"] >= threshold).astype(int)

    tp = int(((llm_bin == 1) & (gold_bin == 1)).sum())
    tn = int(((llm_bin == 0) & (gold_bin == 0)).sum())
    fp = int(((llm_bin == 1) & (gold_bin == 0)).sum())
    fn = int(((llm_bin == 0) & (gold_bin == 1)).sum())

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    return {
        "n": int(len(df)),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "FPR": fpr,
        "FNR": fnr,
        "Precision": precision,
        "Recall": recall,
        "Accuracy": accuracy,
    }


def make_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    out = df.copy()
    for col in ["FPR", "FNR", "Precision", "Recall", "Accuracy", "Delta_FPR", "Delta_FNR"]:
        if col in out.columns:
            out[col] = out[col].map(lambda x: f"{x:.4f}" if pd.notnull(x) else "")

    return out.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
    )

In [7]:
# =========================
# Build FP/FN table for every model/language variant
# =========================
model_files = find_llm_files()
invalid_keys = load_invalid_keys(INVALID_CSV)
print(f"Found {len(model_files)} allowed models under: {LABEL_ROOT}")

rows = []
skipped = 0

for model, files in model_files.items():
    for f in files:
        lang = get_lang_from_filename(f, model)
        if lang not in LANGS:
            continue

        try:
            df = load_labels(f, invalid_keys)
            metrics = compute_fp_fn_metrics(df, RELEVANCE_THRESHOLD)

            rows.append({
                "Model": model,
                "Language": lang,
                "Base_Language": base_lang_from_variant(lang),
                "Variant": variant_type(lang),
                **metrics,
            })

            print(
                f"[INFO] {model} {lang}: n={metrics['n']}, "
                f"FP={metrics['FP']}, FN={metrics['FN']}, "
                f"FPR={metrics['FPR']:.4f}, FNR={metrics['FNR']:.4f}"
            )

        except Exception as e:
            skipped += 1
            print(f"[SKIP] {model} {lang} ({f.name}): {e}")

if not rows:
    raise RuntimeError(
        f"No FP/FN rows produced. Check LABEL_ROOT={LABEL_ROOT}, LANGS={LANGS}, and file schemas."
    )

all_df = pd.DataFrame(rows).sort_values(["Model", "Base_Language", "Variant", "Language"]).reset_index(drop=True)

write_df(all_df, OUT_ALL_CSV)
OUT_ALL_TEX.write_text(
    make_latex_table(
        all_df,
        caption=f"False-positive and false-negative counts/rates by model and language variant, using relevance threshold $\geq {RELEVANCE_THRESHOLD}$.",
        label=f"tab:fp_fn_all_variants_{TREC_DL_YEAR}",
    ),
    encoding="utf-8",
)

print(f"\n[OK] Wrote all-variant FP/FN CSV: {OUT_ALL_CSV}")
print(f"[OK] Wrote all-variant FP/FN TeX: {OUT_ALL_TEX}")
if skipped:
    print(f"[INFO] Skipped {skipped} files due to errors.")

all_df

[INFO] Loaded 6 invalid keys from D:\Work\Research_Project\anaconda_research_project\scripts\report\seaborn_script\statistics_tests\invalid_2022.csv
Found 2 allowed models under: outputs\llm_label\trec_dl_2022
[INFO] gpt-oss-20b ar_distraction_qp_rem: n=2648, FP=526, FN=166, FPR=0.2730, FNR=0.2302
[INFO] gpt-oss-20b ar_qp_rem: n=2649, FP=481, FN=176, FPR=0.2495, FNR=0.2441


<>:50: SyntaxWarning: invalid escape sequence '\g'
<>:50: SyntaxWarning: invalid escape sequence '\g'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16660\4130951419.py:50: SyntaxWarning: invalid escape sequence '\g'
  caption=f"False-positive and false-negative counts/rates by model and language variant, using relevance threshold $\geq {RELEVANCE_THRESHOLD}$.",


[INFO] gpt-oss-20b eng_distraction_qp_rem: n=2647, FP=592, FN=152, FPR=0.3072, FNR=0.2111
[INFO] gpt-oss-20b eng_qp_rem: n=2649, FP=503, FN=143, FPR=0.2609, FNR=0.1983
[INFO] gpt-oss-20b ga_distraction_qp_rem: n=2647, FP=457, FN=214, FPR=0.2373, FNR=0.2968
[INFO] gpt-oss-20b ga_qp_rem: n=2648, FP=458, FN=198, FPR=0.2377, FNR=0.2746
[INFO] gpt-oss-20b he_distraction_qp_rem: n=2648, FP=474, FN=168, FPR=0.2460, FNR=0.2330
[INFO] gpt-oss-20b he_qp_rem: n=2649, FP=480, FN=165, FPR=0.2490, FNR=0.2288
[INFO] gpt-oss-20b raw: n=2649, FP=525, FN=122, FPR=0.2723, FNR=0.1692
[INFO] gpt-oss-20b ru_distraction_qp_rem: n=2649, FP=537, FN=149, FPR=0.2785, FNR=0.2067
[INFO] gpt-oss-20b ru_qp_rem: n=2648, FP=479, FN=169, FPR=0.2486, FNR=0.2344
[INFO] gpt-oss-20b sw_distraction_qp_rem: n=2648, FP=496, FN=180, FPR=0.2574, FNR=0.2497
[INFO] gpt-oss-20b sw_qp_rem: n=2649, FP=479, FN=201, FPR=0.2484, FNR=0.2788
[INFO] gpt-oss-20b th_distraction_qp_rem: n=2647, FP=531, FN=160, FPR=0.2757, FNR=0.2219
[INFO] g

,Model,Language,Base_Language,Variant,n,TP,TN,FP,FN,FPR,FNR,Precision,Recall,Accuracy
0,gpt-oss-20b,ar_distraction_qp_rem,ar,baseline,2648,555,1401,526,166,0.272963,0.230236,0.513414,0.769764,0.738671
1,gpt-oss-20b,ar_qp_rem,ar,baseline,2649,545,1447,481,176,0.249481,0.244105,0.531189,0.755895,0.751982
2,gpt-oss-20b,eng_distraction_qp_rem,eng,baseline,2647,568,1335,592,152,0.307213,0.211111,0.489655,0.788889,0.718927
3,gpt-oss-20b,eng_qp_rem,eng,baseline,2649,578,1425,503,143,0.260892,0.198336,0.534690,0.801664,0.756134
4,gpt-oss-20b,ga_distraction_qp_rem,ga,baseline,2647,507,1469,457,214,0.237279,0.296810,0.525934,0.703190,0.746505
5,gpt-oss-20b,ga_qp_rem,ga,baseline,2648,523,1469,458,198,0.237675,0.274619,0.533129,0.725381,0.752266
6,gpt-oss-20b,he_distraction_qp_rem,he,baseline,2648,553,1453,474,168,0.245978,0.233010,0.538462,0.766990,0.757553
7,gpt-oss-20b,he_qp_rem,he,baseline,2649,556,1448,480,165,0.248963,0.228849,0.536680,0.771151,0.756512
8,gpt-oss-20b,raw,raw,other,2649,599,1403,525,122,0.272303,0.169209,0.532918,0.830791,0.755757
9,gpt-oss-20b,ru_distraction_qp_rem,ru,baseline,2649,572,1391,537,149,0.278527,0.206657,0.515780,0.793343,0.741034


In [8]:
# =========================
# Build baseline-vs-category comparison table
# =========================
# This table is the most useful for the paper because it directly measures
# how FP/FN changed from the baseline variant to the attacked/defended category variant.

base_df = all_df[all_df["Variant"] == "baseline"].copy()
cat_df = all_df[all_df["Variant"] == "category"].copy()

compare_df = base_df.merge(
    cat_df,
    on=["Model", "Base_Language"],
    suffixes=("_baseline", "_category"),
    how="inner",
)

if compare_df.empty:
    print("[WARN] No matched baseline/category pairs were found. Check BASELINE_SUFFIX, CATEGORY_SUFFIX, and LANG_PROFILE.")
else:
    compare_df = compare_df[[
        "Model",
        "Base_Language",
        "Language_baseline",
        "Language_category",
        "n_baseline",
        "n_category",
        "FP_baseline",
        "FP_category",
        "FN_baseline",
        "FN_category",
        "FPR_baseline",
        "FPR_category",
        "FNR_baseline",
        "FNR_category",
    ]].copy()

    compare_df["Delta_FP"] = compare_df["FP_category"] - compare_df["FP_baseline"]
    compare_df["Delta_FN"] = compare_df["FN_category"] - compare_df["FN_baseline"]
    compare_df["Delta_FPR"] = compare_df["FPR_category"] - compare_df["FPR_baseline"]
    compare_df["Delta_FNR"] = compare_df["FNR_category"] - compare_df["FNR_baseline"]

    # Ratio is useful but can be unstable when FP_baseline is 0.
    compare_df["FP_ratio"] = compare_df.apply(
        lambda r: r["FP_category"] / r["FP_baseline"] if r["FP_baseline"] > 0 else pd.NA,
        axis=1,
    )

    compare_df = compare_df.sort_values(["Model", "Base_Language"]).reset_index(drop=True)

    write_df(compare_df, OUT_COMPARE_CSV)
    OUT_COMPARE_TEX.write_text(
        make_latex_table(
            compare_df,
            caption=(
                f"Baseline-vs-category false-positive and false-negative comparison by model and language, "
                f"using relevance threshold $\geq {RELEVANCE_THRESHOLD}$. Positive $\Delta FP$ indicates false-positive inflation."
            ),
            label=f"tab:fp_fn_comparison_{TREC_DL_YEAR}",
        ),
        encoding="utf-8",
    )

    print(f"[OK] Wrote FP/FN comparison CSV: {OUT_COMPARE_CSV}")
    print(f"[OK] Wrote FP/FN comparison TeX: {OUT_COMPARE_TEX}")

compare_df

[WARN] No matched baseline/category pairs were found. Check BASELINE_SUFFIX, CATEGORY_SUFFIX, and LANG_PROFILE.


<>:56: SyntaxWarning: invalid escape sequence '\g'
<>:56: SyntaxWarning: invalid escape sequence '\D'
<>:56: SyntaxWarning: invalid escape sequence '\g'
<>:56: SyntaxWarning: invalid escape sequence '\D'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16660\2395326471.py:56: SyntaxWarning: invalid escape sequence '\g'
  f"using relevance threshold $\geq {RELEVANCE_THRESHOLD}$. Positive $\Delta FP$ indicates false-positive inflation."
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16660\2395326471.py:56: SyntaxWarning: invalid escape sequence '\D'
  f"using relevance threshold $\geq {RELEVANCE_THRESHOLD}$. Positive $\Delta FP$ indicates false-positive inflation."


,Model,Language_baseline,Base_Language,Variant_baseline,n_baseline,TP_baseline,TN_baseline,FP_baseline,FN_baseline,FPR_baseline,...,n_category,TP_category,TN_category,FP_category,FN_category,FPR_category,FNR_category,Precision_category,Recall_category,Accuracy_category


In [9]:
# Optional: compact summary by model
# This helps quickly see whether the category variant increases FP overall.

if not compare_df.empty:
    model_summary = (
        compare_df
        .groupby("Model", as_index=False)
        .agg(
            Mean_Delta_FP=("Delta_FP", "mean"),
            Median_Delta_FP=("Delta_FP", "median"),
            Mean_Delta_FN=("Delta_FN", "mean"),
            Median_Delta_FN=("Delta_FN", "median"),
            Mean_Delta_FPR=("Delta_FPR", "mean"),
            Mean_Delta_FNR=("Delta_FNR", "mean"),
        )
    )
    display(model_summary)